In [ ]:
# Cell 1 — Setup and imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import subprocess

plt.style.use('seaborn-v0_8-whitegrid')

# Make research/ importable when running from research/
sys.path.insert(0, os.path.abspath('.'))

from ou_calibration import calibrate_ou
from ofi_signal    import analyze_ofi
from frontier      import run_frontier, plot_frontier

os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../data/processed',  exist_ok=True)
print('Setup complete.')

## Project Overview

### Friction-Aware Delta Hedging

**Problem.** Every options market maker delta-hedges to stay directionally neutral. Black-Scholes assumes this is free and continuous. In practice every hedge crosses the bid-ask spread, paying a cost proportional to shares traded and half the spread. This creates a fundamental tension:

- Hedge **too often** → transaction costs grind down P&L
- Hedge **too rarely** → unhedged gamma exposure builds up

**Data.** LOBSTER AAPL Level-10 order book data, June 21 2012 (~400k tick events). AAPL daily closing prices 2010–2012 via yfinance for OU calibration.

**Core hedge inequality:**

$$\text{Hedge} \iff \underbrace{\frac{1}{2}\,|\Gamma|\, S^2\, \sigma^2\, \Delta t}_{\text{gamma risk from waiting}} > \lambda \times \underbrace{\Delta_{\text{shares}} \times \frac{\text{spread}}{2}}_{\text{cost of hedging now}}$$

**Three findings:** (1) U-shaped P&L frontier — an optimal λ minimises total cost. (2) OFI-timed hedges achieve lower realized slippage. (3) Empirical validation of Leland (1985).

In [ ]:
# Cell 3 — Load and summarise simulation output
tick_log  = pd.read_csv('../data/processed/tick_log.csv')
hedge_log = pd.read_csv('../data/processed/hedge_log.csv')
summary   = pd.read_csv('../data/processed/summary.csv')

s = summary.iloc[0]

avg_spread_bps = (tick_log['spread'] / tick_log['mid_price'] * 10000).mean()

log_rets    = np.log(tick_log['mid_price'] / tick_log['mid_price'].shift(1)).dropna()
avg_dt      = tick_log['timestamp'].diff().mean()
rv_intraday = log_rets.std() * np.sqrt(252 * 6.5 * 3600 / avg_dt)

print('=== Simulation Summary (lambda = 1.0) ===')
print(f'  Tick events processed      : {int(s["total_events"]):>10,}')
print(f'  Hedges executed            : {int(s["total_hedges"]):>10,}')
print(f'  Avg bid-ask spread         : ${s["avg_spread"]:>10.4f}  ({avg_spread_bps:.1f} bps)')
print(f'  Avg time between hedges    : {s["avg_time_between_hedges"]:>10.1f} s')
print(f'  Cumulative theta P&L       : ${s["total_theta_pnl"]:>+10.2f}')
print(f'  Cumulative gamma P&L       : ${s["total_gamma_pnl"]:>+10.2f}')
print(f'  Total transaction costs    : ${s["total_tcost"]:>10.2f}')
print(f'  Net P&L                    : ${s["net_pnl"]:>+10.2f}')
print()
print('=== AAPL Intraday Statistics ===')
print(f'  Opening mid price          : ${tick_log["mid_price"].iloc[0]:>10.4f}')
print(f'  Closing mid price          : ${tick_log["mid_price"].iloc[-1]:>10.4f}')
print(f'  Intraday range             :  ${tick_log["mid_price"].min():.4f} / ${tick_log["mid_price"].max():.4f}')
print(f'  Total price move           : ${abs(tick_log["mid_price"].iloc[-1] - tick_log["mid_price"].iloc[0]):>10.4f}')
print(f'  Realized vol (annualised)  : {rv_intraday:>10.4f}  ({rv_intraday*100:.1f}%)')

In [ ]:
# Cell 4 — OU Calibration
ou_params = calibrate_ou(save_fig=True)

print('\n=== OU Parameter Table ===')
param_df = pd.DataFrame({
    'Parameter': ['kappa', 'theta', 'xi'],
    'Value':     [ou_params['kappa'], ou_params['theta'], ou_params['xi']],
    'Interpretation': [
        f"Mean reversion speed — half-life {np.log(2)/ou_params['kappa']*252:.0f} trading days",
        f"Long-run mean vol = {ou_params['theta']*100:.1f}%  (simulation uses 25%)",
        f"Vol of vol = {ou_params['xi']:.4f}",
    ]
})
print(param_df.to_string(index=False))

from IPython.display import Image, display
display(Image('../results/figures/ou_calibration.png'))

**Interpretation.** A positive κ confirms mean reversion: volatility does not random-walk indefinitely but reverts toward θ after spikes. The half-life tells you how quickly. θ being close to 25% means the engine is operating near the long-run neutral vol regime. ξ quantifies vol-of-vol: higher ξ means the future vol path is less certain and the optimal hedge bandwidth should be wider.

In [ ]:
# Cell 5 — OFI Timing Analysis
ofi_results = analyze_ofi(save_fig=True)

fav   = ofi_results['favorable']
unfav = ofi_results['unfavorable']

print('\n=== OFI Results ===')
res_df = pd.DataFrame({
    'Metric':          ['Count', 'Mean slip/share ($)', 'Median slip/share ($)', 'Total slippage ($)'],
    'Favorable OFI':   [fav['count'],   f"{fav['mean']:.4f}",   f"{fav['median']:.4f}",   f"{fav['total']:.4f}"],
    'Unfavorable OFI': [unfav['count'], f"{unfav['mean']:.4f}", f"{unfav['median']:.4f}", f"{unfav['total']:.4f}"],
})
print(res_df.to_string(index=False))
print(f"\nSlippage improvement : {ofi_results['pct_improvement']:.1f}%")
print(f"t-statistic          : {ofi_results['t_statistic']:.3f}")
print(f"p-value              : {ofi_results['p_value']:.4f}")

from IPython.display import Image, display
display(Image('../results/figures/ofi_analysis.png'))

**Interpretation.** When the engine executes hedges with order flow aligned to the trade direction, it achieves lower slippage per share. This is a pure execution-timing effect — the hedge decision itself is unchanged. Even a small per-share improvement compounds significantly across hundreds of hedges per day and across a portfolio of many positions.

In [ ]:
# Cell 6 — Run Efficient Frontier (~1–2 minutes)
print('Running frontier sweep — this will take ~1-2 minutes...\n')
frontier_df = run_frontier(
    ob_file='../data/raw/AAPL-Lobster-Orderbook.csv',
    msg_file='../data/raw/AAPL-Lobster-Message.csv',
    engine_path='../engine/hedger',
)

print('\n=== Frontier Results ===')
display_cols = ['lambda_val', 'total_hedges', 'net_pnl',
                'total_tcost', 'total_theta_pnl', 'total_gamma_pnl', 'delta_gap_variance']
print(frontier_df[display_cols].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

plot_frontier(frontier_df, save_fig=True)

from IPython.display import Image, display
display(Image('../results/figures/efficient_frontier.png'))

## Key Findings

---

### Finding 1 — The U-Shaped P&L Frontier

Net P&L is not monotone in hedging frequency. At very low λ (aggressive hedging) transaction costs dominate. As λ increases, costs fall faster than gamma losses rise, improving net P&L up to an optimal point. Beyond that optimal λ, unhedged gamma accumulates faster than cost savings justify, and net P&L deteriorates. The optimal λ — marked on Panel 2 of the frontier figure — is the point where marginal gamma risk saved equals marginal transaction cost paid. This is the empirical core of **Leland (1985)**.

---

### Finding 2 — OFI Execution Improvement

Hedges executed with OFI aligned to the trade direction achieve lower average slippage per share than hedges executed without regard to OFI. The improvement is consistent across buy and sell hedges and is statistically testable. For a market maker running a large book, this execution edge applied consistently reduces total transaction drag meaningfully over a trading day.

---

### Finding 3 — Leland (1985) Validated on Real Microstructure Data

The efficient frontier traces a curve in transaction-cost / hedging-error space consistent with Leland’s theoretical prediction: reducing hedging error toward zero requires exponentially increasing transaction costs. The tradeoff is not linear — the first reduction in hedging error is cheap, but approaching delta-neutral continuously becomes prohibitively expensive. This validates the model using real NASDAQ order-book data rather than a theoretical cost model.

**Reference:** Leland, H. E. (1985). *Option Pricing and Replication with Transactions Costs.* Journal of Finance, 40(5), 1283–1301.

In [ ]:
# Cell 8 — Four-panel summary figure
plt.style.use('seaborn-v0_8-whitegrid')

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.30)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

# Subsample tick_log to ~3000 points for clean line plots
step = max(1, len(tick_log) // 3000)
tl   = tick_log.iloc[::step].copy()
hrs  = (tl['timestamp'] - 34200) / 3600

# Panel 1: AAPL mid price
ax1.plot(hrs, tl['mid_price'], color='steelblue', linewidth=0.8)
ax1.set_title('AAPL Mid Price — June 21, 2012')
ax1.set_xlabel('Hours since open (09:30)')
ax1.set_ylabel('Mid Price ($)')
ax1.set_xlim(0, 6.5)

# Panel 2: cumulative P&L components
net_cum = tl['cumulative_gamma_pnl'] + tl['cumulative_theta_pnl'] - tl['cumulative_tcost']
ax2.plot(hrs, tl['cumulative_gamma_pnl'],  color='tomato',         linewidth=1.0, label='Gamma P&L')
ax2.plot(hrs, tl['cumulative_theta_pnl'],  color='mediumseagreen', linewidth=1.0, label='Theta P&L')
ax2.plot(hrs, net_cum,                     color='steelblue',      linewidth=1.3, linestyle='--', label='Net P&L')
ax2.axhline(0, color='black', linewidth=0.6, linestyle=':')
ax2.set_title('Cumulative P&L Components')
ax2.set_xlabel('Hours since open (09:30)')
ax2.set_ylabel('Cumulative P&L ($)')
ax2.set_xlim(0, 6.5)
ax2.legend(fontsize=9)

# Panel 3: efficient frontier
try:
    fr = pd.read_csv('../data/processed/frontier_results.csv')
    ax3.plot(fr['total_tcost'], fr['delta_gap_variance'],
             'o-', color='steelblue', linewidth=1.5, markersize=7)
    for _, row in fr.iterrows():
        ax3.annotate(f"\u03bb={row['lambda_val']:.2g}",
                     xy=(row['total_tcost'], row['delta_gap_variance']),
                     xytext=(5, 3), textcoords='offset points',
                     fontsize=7, color='dimgray')
    ax3.annotate('\u2190 optimal region', xy=(0.04, 0.06), xycoords='axes fraction',
                 fontsize=8, color='darkgreen', style='italic')
except FileNotFoundError:
    ax3.text(0.5, 0.5, 'Run Cell 6 first\n(frontier_results.csv not found)',
             ha='center', va='center', transform=ax3.transAxes, color='dimgray')
ax3.set_title('Efficient Hedging Frontier')
ax3.set_xlabel('Total Transaction Costs ($)')
ax3.set_ylabel('Variance of Delta Gap (shares\u00b2)')

# Panel 4: spread + hedge events
ax4.plot(hrs, tl['spread'], color='dimgray', linewidth=0.6, alpha=0.8)
hedge_rows = tick_log[tick_log['hedged'] == 1]
hedge_hrs  = (hedge_rows['timestamp'] - 34200) / 3600
for h in hedge_hrs:
    ax4.axvline(h, color='tomato', alpha=0.25, linewidth=0.6)
if len(hedge_hrs) > 0:
    ax4.axvline(hedge_hrs.iloc[0], color='tomato', alpha=0.7, linewidth=0.8,
                label=f'Hedge events (n={len(hedge_hrs)})')
ax4.set_title('Bid-Ask Spread + Hedge Events')
ax4.set_xlabel('Hours since open (09:30)')
ax4.set_ylabel('Spread ($)')
ax4.set_xlim(0, 6.5)
ax4.legend(fontsize=9)

fig.suptitle('Friction-Aware Delta Hedging  —  AAPL June 21 2012  (λ = 1.0)',
             fontsize=13, fontweight='bold', y=1.01)

path = '../results/figures/summary_figure.png'
fig.savefig(path, dpi=150, bbox_inches='tight')
print(f'Summary figure saved → {path}')
plt.show()